# 米国株AIレポート MVP（Colabで1回動かす版）

このノートブックは、以下を1回で実行します。

1. Google Driveをマウント
2. 必要ライブラリをインストール
3. YouTube検索・RSSから公開情報を取得
4. Jim Cramer / Ray Dalio / Scott Galloway 関連情報を整理
5. 米国株ティッカーを簡易抽出
6. yfinanceで市場データを取得
7. PDFレポートをGoogle Driveへ保存
8. Gmail SMTPで指定メールアドレスへ送信

注意：
- 初期MVPなので、外部LLM APIは使いません。
- YouTube音声のダウンロードは行いません。字幕・説明文・タイトルを優先します。
- 本レポートは投資助言ではなく、公開情報を整理した調査資料です。

In [ ]:
# セル1：Google Driveをマウントし、必要ライブラリを入れます
from google.colab import drive
drive.mount('/content/drive')

!apt-get -qq update > /dev/null
!apt-get -qq install -y fonts-noto-cjk nodejs npm > /dev/null
!pip -q install -U yfinance feedparser beautifulsoup4 requests reportlab pandas yt-dlp

print("準備完了：Driveマウントとライブラリ導入が終わりました。")


In [ ]:
# セル2：あなたの情報を入力します
# Gmailの通常パスワードではなく「アプリパスワード」を入れてください。
# 入力されたパスワードは画面に表示されません。

import getpass
import os
from datetime import datetime

DRIVE_ROOT = "/content/drive/MyDrive/ai-investment-agent"
REPORT_DIR = os.path.join(DRIVE_ROOT, "reports")
LOG_DIR = os.path.join(DRIVE_ROOT, "logs")
DATA_DIR = os.path.join(DRIVE_ROOT, "data")
ASSETS_DIR = os.path.join(DRIVE_ROOT, "assets")

for p in [DRIVE_ROOT, REPORT_DIR, LOG_DIR, DATA_DIR, ASSETS_DIR]:
    os.makedirs(p, exist_ok=True)

print("保存先:", DRIVE_ROOT)
print("表紙画像フォルダ:", ASSETS_DIR)

EMAIL_TO = input("レポートを受け取るメールアドレスを入力してください: ").strip()
SMTP_USER = input("送信用Gmailアドレスを入力してください: ").strip()
EMAIL_FROM = SMTP_USER
SMTP_PASSWORD = getpass.getpass("Gmailアプリパスワード16桁を入力してください: ").strip()

send_choice = input("PDFをメール送信しますか？ y/n [y]: ").strip().lower()
SEND_EMAIL = (send_choice != "n")
dry_choice = input("dry-runで実行しますか？ y/n [n]: ").strip().lower()
DRY_RUN = (dry_choice == "y")

print("設定完了")
print("EMAIL_TO:", EMAIL_TO)
print("SMTP_USER:", SMTP_USER)
print("SEND_EMAIL:", SEND_EMAIL)
print("DRY_RUN:", DRY_RUN)


In [ ]:
# セル3：MVP本体。ここを実行すると、情報取得→分析→PDF作成→メール送信まで行います。
import os,re,html,math,time,smtplib,logging,traceback,subprocess
from email.message import EmailMessage
from datetime import datetime
from collections import defaultdict
from xml.sax.saxutils import escape
import feedparser,pandas as pd,yfinance as yf
from bs4 import BeautifulSoup
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.cidfonts import UnicodeCIDFont
import yt_dlp

COVER_IMAGE_PROMPT = """A bright, exciting, photorealistic A4 landscape cover image for an AI-powered US stock research report. A futuristic but clean investment research room with a large transparent AI dashboard in the center. The dashboard displays elegant US stock charts, market news cards, YouTube video cards, financial data panels, risk indicators, and glowing opportunity signals. The scene suggests that AI is analyzing public information from financial news, YouTube commentary, macroeconomic data, and company fundamentals to identify potential growth stocks and risk candidates for the next two years. The mood is optimistic, intelligent, professional, clean, and inspiring. Use bright white, blue, and gold tones. Photorealistic, high detail, modern financial technology atmosphere. No people. Minimal readable text. No real company logos. No financial advice wording. Suitable as the first-page cover of a professional Japanese investment research PDF."""
# 日本語版: A4横向きの明るく知的な写真風ビジュアル。未来的で清潔感のあるAI投資リサーチ室。中央に透明な大型AIダッシュボードがあり、米国株チャート、マーケットニュースカード、YouTube動画カード、財務データパネル、リスク警告アイコン、上昇候補を示す光のサインが美しく整理されている。AIが金融ニュース、YouTube解説、マクロ経済情報、企業財務データなどの公開情報を分析し、今後2年の成長候補と注意候補を見つけているイメージ。雰囲気は明るい、わくわく、知的、プロフェッショナル、清潔。白、青、金を基調。写真のようにリアル。人物なし。実在企業ロゴなし。文字は最小限。投資助言ではなく、日本語の投資調査レポートの表紙にふさわしい画像。

CONFIG={"report":{"language":"ja","title":"米国株AI投資調査レポート","page_size":"A4","orientation":"landscape","include_cover_image":True,"cover_image_path":f"{ASSETS_DIR}/cover.png","include_summary":True,"include_metric_definitions":True,"top_candidates_count":5,"avoid_candidates_count":5,"include_disclaimer":True,"include_source_urls":True},"design":{"theme":"bright_professional","primary_color":"#1E5BFF","accent_color":"#F2B705","background_color":"#F7FAFF","card_background":"#FFFFFF"}}

today=datetime.now().strftime("%Y-%m-%d")
log_path=os.path.join(LOG_DIR,f"daily_{today}.log")
logging.basicConfig(level=logging.INFO,format="%(asctime)s [%(levelname)s] %(message)s",handlers=[logging.FileHandler(log_path,encoding="utf-8"),logging.StreamHandler()])
logger=logging.getLogger("mvp")

YOUTUBE_SEARCHES=[{"speaker":"Jim Cramer","query":"ytsearch4:Jim Cramer Mad Money stock picks today CNBC"},{"speaker":"Ray Dalio","query":"ytsearch3:Ray Dalio markets economy debt cycle latest"},{"speaker":"Scott Galloway","query":"ytsearch3:Scott Galloway AI big tech stocks latest"}]
RSS_FEEDS=[{"name":"Yahoo Finance","url":"https://finance.yahoo.com/news/rssindex"}]
TICKERS={"NVDA":"NVIDIA","MSFT":"Microsoft","AAPL":"Apple","AMZN":"Amazon","GOOGL":"Alphabet","META":"Meta","TSLA":"Tesla","AMD":"AMD"}

def clean_text(t): return re.sub(r"\s+"," ",BeautifulSoup(str(t),"html.parser").get_text(" ")).strip()
def truncate(t,n=220): return t if len(t)<=n else t[:n]+"..."
def pct(v): return "N/A" if v is None or pd.isna(v) else f"{v*100:.1f}%"
def get_market_data(t):
    try:
        h=yf.Ticker(t).history(period="6mo"); info=yf.Ticker(t).info
        p=float(h['Close'].iloc[-1]) if len(h) else None; r=(h['Close'].iloc[-1]/h['Close'].iloc[0]-1) if len(h)>1 else None
        return {"price":p,"return_6m":r,"pe":info.get("trailingPE")}
    except: return {"price":None,"return_6m":None,"pe":None}

def ensure_node_runtime():
    try:
        subprocess.run(["node","--version"],check=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE)
        return True
    except Exception:
        return False

def collect_youtube_items():
    items=[]
    ydl_opts={"quiet":True,"no_warnings":True,"skip_download":True,"noplaylist":True,"ignoreerrors":True,"socket_timeout":20,"extract_flat":False}
    if ensure_node_runtime():
        ydl_opts["extractor_args"]={"youtube":{"player_client":["android","web"]}}
    for s in YOUTUBE_SEARCHES:
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl: info=ydl.extract_info(s['query'],download=False)
            for e in (info or {}).get("entries",[]) or []:
                if not e: continue
                title=clean_text(e.get("title","")); url=e.get("webpage_url") or e.get("url") or ""
                desc=clean_text(e.get("description","") or "")
                text=clean_text(" ".join([title,desc]))
                status="ok" if len(text)>=40 else "text_insufficient"
                confidence="medium" if status=="ok" else "low"
                items.append({"source_type":"youtube","speaker":s['speaker'],"title":title,"url":url,"channel":e.get("channel") or e.get("uploader") or "","published":e.get("upload_date","") ,"description":desc,"thumbnail_url":e.get("thumbnail","") ,"text":text,"status":status,"confidence":confidence})
        except Exception as ex:
            logger.warning(f"YouTube取得失敗({s['speaker']}): {ex}")
    return items

def collect_rss_items():
    items=[]
    for f in RSS_FEEDS:
        d=feedparser.parse(f['url'])
        for e in d.entries[:8]: items.append({"source_type":"rss","speaker":"Market News","title":clean_text(e.get('title','')),"url":e.get('link',''),"published":e.get('published',''),"description":clean_text(e.get('summary','')),"text":clean_text(e.get('title','')+" "+e.get('summary','')),"status":"ok","confidence":"medium"})
    return items

def score_rows(items):
    rows=[]
    for t,c in TICKERS.items():
        txt=" ".join([i.get('text','') for i in items]).lower(); m=txt.count(t.lower())+txt.count(c.lower().split()[0].lower())
        md=get_market_data(t); up=min(95,45+m*8+(12 if (md['return_6m'] or 0)>0.15 else 0)); risk=min(95,35+(18 if (md['pe'] or 0)>45 else 0)+(8 if (md['return_6m'] or 0)<0 else 0))
        rows.append({"ticker":t,"company":c,"upside_score":int(up),"avoid_score":int(risk),"reason":"複数ソース言及と市場データを総合評価。","risk":"バリュエーションやボラティリティに注意。","market":md})
    return rows

def build_pdf(items,rows):
    pdfmetrics.registerFont(UnicodeCIDFont("HeiseiKakuGo-W5"))
    path=os.path.join(REPORT_DIR,f"us_stock_ai_report_{today}.pdf")
    doc=SimpleDocTemplate(path,pagesize=landscape(A4),rightMargin=10*mm,leftMargin=10*mm,topMargin=10*mm,bottomMargin=10*mm)
    st=getSampleStyleSheet(); base=ParagraphStyle("b",parent=st['Normal'],fontName="HeiseiKakuGo-W5",fontSize=9,leading=12)
    h1=ParagraphStyle("h1",parent=base,fontSize=16,textColor=colors.HexColor("#1E5BFF")); h2=ParagraphStyle("h2",parent=base,fontSize=12)
    def P(t,s=base): return Paragraph(escape(str(t)).replace("\n","<br/>"),s)
    top=sorted(rows,key=lambda x:x['upside_score'],reverse=True)[:3]; avoid=sorted(rows,key=lambda x:x['avoid_score'],reverse=True)[:3]
    story=[P(CONFIG['report']['title'],h1),P(f"作成日: {today}"),P("本レポートは投資助言ではなく、公開情報に基づく調査メモです。"),Spacer(1,4)]
    story += [P("本日の結論",h2),P("市場全体の印象: AI・半導体・大型テックへの関心は継続。金利と割高感は注意。"),P("今日の最大チャンス: AI関連の実需拡大。"),P("今日の最大リスク: 高バリュエーション銘柄の変動。"),P("Cramer視点: 短期テーマ重視。 Dalio視点: マクロ循環重視。 Galloway視点: 長期競争優位重視。")]
    story += [P("上昇候補トップ3: "+", ".join([r['ticker'] for r in top])),P("注意候補トップ3: "+", ".join([r['ticker'] for r in avoid]))]
    story.append(PageBreak())
    story += [P("サマリー",h1),P(f"取得ソース数: YouTube {sum(1 for i in items if i['source_type']=='youtube')} / RSS {sum(1 for i in items if i['source_type']=='rss')}"),P("このレポートで使う指標の意味",h2),P("Upside Score: 今後2年の上昇可能性を100点で示す参考指標。"),P("Risk Score: 変動性や割高感などの注意指標。"),P("Momentum: 直近の株価の勢い。 Valuation: 割高/割安の見方。 Catalyst: 株価を動かす材料。"),P("Source Confidence: 情報源の信頼度。 Cramer Dependence: Cramer発言への依存度。 Macro Fit: マクロ追い風適合度。")]
    story.append(PageBreak())
    for sec,title_rows in [("上昇候補詳細",top),("注意候補詳細",avoid)]:
        story.append(P(sec,h1))
        for r in title_rows:
            story += [P(f"{r['ticker']} ({r['company']})",h2),P(f"Upside Score {r['upside_score']}: 今後2年の上昇余地を100点満点で示した参考指標。"),P(f"Risk Score {r['avoid_score']}: 値動きの荒さや割高感などを示す注意指標。"),P(f"理由1行: {r['reason']} ただし {r['risk']}")]
        story.append(PageBreak())
    for name in ["Jim Cramer視点","Ray Dalio視点","Scott Galloway視点"]:
        story += [P(name,h1),P("本日は十分な新規情報が取得できなかったため、参考情報として扱います。")]
    story += [PageBreak(),P("免責事項",h1),P("本レポートは公開情報をもとにAIが自動生成した調査資料であり、投資助言ではありません。最終的な投資判断は利用者自身の責任で行ってください。記載内容には誤り、遅延、不完全な情報が含まれる可能性があります。")]
    doc.build(story); return path

def send_email_with_pdf(path):
    if DRY_RUN:
        logger.info("dry-runのためメール送信はスキップしました。")
        return False
    if not SEND_EMAIL: return False
    msg=EmailMessage(); msg['Subject']=f"【米国株AIレポート】{today}版"; msg['From']=EMAIL_FROM; msg['To']=EMAIL_TO
    msg.set_content("米国株AI投資調査レポートを添付します。")
    with open(path,'rb') as f: msg.add_attachment(f.read(),maintype='application',subtype='pdf',filename=os.path.basename(path))
    with smtplib.SMTP('smtp.gmail.com',587,timeout=40) as s: s.starttls(); s.login(SMTP_USER,SMTP_PASSWORD); s.send_message(msg)
    return True

try:
    items=collect_youtube_items()+collect_rss_items()
    pdf_path=build_pdf(items,score_rows(items))
    sent=send_email_with_pdf(pdf_path)
    if DRY_RUN: print("dry-runのためメール送信はスキップしました。\nPDFは以下に保存されました：\n"+pdf_path)
    print("ログ保存先:",log_path)
except Exception:
    logger.error(traceback.format_exc())
    print("エラー。ログ確認:",log_path)


## 次にやること

1回目が成功したら、次は検索クエリや対象銘柄を増やします。  
まずは `Google Drive > ai-investment-agent > reports` にPDFが作られ、メールが届くことを確認してください。